# topocorr example: Kalam (upper Swat, Hindu Kush), Landsat 5 and Landsat 8-9

Is a forest trend on these slopes real, or illumination? This notebook

1. fits one stack model per sensor family (Landsat 5, 1990-2012; Landsat 8-9, 2013-2025);
2. maps the drift artefact and risk class of the NIRv trend with `diagnose` (Landsat 5 by default);
3. compares the correction methods with `compare_export` + `compare` (Landsat 8-9);
4. draws the figures with eofigs.

**Earth Engine use** (Community tier: 150 EECU-hours a month). Measured on this 24 km tile:

| step | EECU-hours | output |
|---|---|---|
| `fit`, each sensor family | not measured (interactive requests) | model JSON |
| `diagnose` map, Landsat 5 (172 images) | 1.7-3.9 (two runs) | GeoTIFF, about 20 MB |
| `diagnose` map, Landsat 8-9 (269 images; off by default) | about 6.1 | GeoTIFF, about 20 MB |
| `compare_export`, Landsat 8-9 (5,761 points) | 0.6-2.4 (two runs) | CSV, 113 MB |

**Resumable.** Every step checks its output first: a model file, a started task (recorded in
`tasks.json`) or a downloaded file is never made twice. Re-run the notebook from the top after an
interruption. `fit` keeps a cache and continues where it stopped.

Run with `topocorr`, `eofigs==0.2.0`, `rioxarray` and `tqdm` installed (see `examples/README.md`).

In [ ]:
# Settings
from pathlib import Path

PROJECT = "my-ee-project"  # your Earth Engine Cloud project
SITE = "kalam"  # prefix of every file and task name; change it for a new run
OUT = Path(f"{SITE}_outputs")  # models, task records and downloaded files
DRIVE_FOLDER = "topocorr_exports"  # Google Drive folder of the exports
RCLONE_REMOTE = None  # e.g. "gdrive:topocorr_exports/" to download with rclone
CENTER = (72.574, 35.525)  # lon, lat
HALF_WIDTH_M = 12000  # 24 km square; 6000 for a 12 km sub-tile (about 1/4 of the cost)
WRS = (151, 35)  # one Landsat path/row: one pixel grid
MAX_CLOUD = 40
INDEX = "nirv"  # drift checks: NIRv or EVI (NDVI cancels part of the artefact)
MAP_OLI = False  # also map Landsat 8-9 (about 6.1 EECU-hours)

if PROJECT.strip() in ("", "my-ee-project"):
    raise ValueError(
        "Set PROJECT to your Earth Engine Cloud project id in this cell, then run again."
    )

In [ ]:
import json
import shutil
import subprocess
import time
from pathlib import Path

import ee
import numpy as np
from tqdm.auto import tqdm

import topocorr

ee.Initialize(project=PROJECT)
OUT.mkdir(parents=True, exist_ok=True)
TASKS = OUT / "tasks.json"  # every Earth Engine task started by this notebook, by name


def load_tasks():
    return json.loads(TASKS.read_text(encoding="utf-8")) if TASKS.exists() else {}


def start_once(name, make_task):
    """Start a batch task unless one with this name already ran or is running (resumable)."""
    tasks = load_tasks()
    if name in tasks and tasks[name]["state"] not in ("FAILED", "CANCELLED"):
        print(f"{name}: already started ({tasks[name]['state']}), skipped")
        return
    task = make_task()
    task.start()
    tasks[name] = {"id": task.id, "state": "READY"}
    TASKS.write_text(json.dumps(tasks, indent=1), encoding="utf-8")
    print(f"{name}: started task {task.id}")


def wait_for_tasks(poll=60):
    """Progress bar over the started tasks until all have finished; reports EECU-hours."""
    tasks = load_tasks()
    done_states = ("COMPLETED", "FAILED", "CANCELLED")
    with tqdm(total=len(tasks), desc="Earth Engine tasks", unit="task") as bar:
        while True:
            eecu = 0.0
            for t in tasks.values():
                s = ee.data.getTaskStatus(t["id"])[0]
                t["state"] = s["state"]
                t["eecu_hours"] = round((s.get("batch_eecu_usage_seconds") or 0) / 3600, 3)
                t["error"] = s.get("error_message", "")
                eecu += t["eecu_hours"]
            TASKS.write_text(json.dumps(tasks, indent=1), encoding="utf-8")
            finished = sum(t["state"] in done_states for t in tasks.values())
            bar.n = finished
            bar.set_postfix(eecu_h=f"{eecu:.2f}")
            bar.refresh()
            if finished == len(tasks):
                break
            time.sleep(poll)
    for name, t in tasks.items():
        print(f"{name}: {t['state']}, {t['eecu_hours']} EECU-h {t['error']}")


def fetch(files):
    """Copy finished exports from Drive with rclone (if RCLONE_REMOTE is set), else list them."""
    missing = [f for f in files if not (OUT / f).exists()]
    if not missing:
        print("all files are present in", OUT)
        return
    if RCLONE_REMOTE and shutil.which("rclone"):
        for f in tqdm(missing, desc="download", unit="file"):
            subprocess.run(["rclone", "copy", f"{RCLONE_REMOTE}{f}", str(OUT)], check=True)
    else:
        print(f"Download these files from the Google Drive folder {DRIVE_FOLDER!r} into {OUT}:")
        for f in missing:
            print("  ", f)


def scene_grid(images):
    """CRS and transform of the first image's 30 m grid: maps line up with the Landsat pixels."""
    p = ee.Image(images.first()).select("nir").projection().getInfo()
    return p["crs"], p["transform"]

## 1. Collections and the tile
One WRS path/row keeps every image on the same pixel grid.

In [ ]:
center = ee.Geometry.Point(CENTER)
utm = f"EPSG:{32600 + int((CENTER[0] + 180) // 6) + 1}"
x, y = center.transform(utm, 1).coordinates().getInfo()
x, y = round(x / 30) * 30, round(y / 30) * 30
aoi = ee.Geometry.Rectangle(
    [x - HALF_WIDTH_M, y - HALF_WIDTH_M, x + HALF_WIDTH_M, y + HALF_WIDTH_M],
    ee.Projection(utm),
    False,
)


def landsat(sensors, start, end):
    col = None
    for s in sensors:
        c = topocorr.collection(s, aoi, start, end, max_cloud=MAX_CLOUD).filter(
            ee.Filter.And(ee.Filter.eq("WRS_PATH", WRS[0]), ee.Filter.eq("WRS_ROW", WRS[1]))
        )
        col = c if col is None else col.merge(c)
    return col.sort("system:time_start")


families = {
    "l5": landsat(["L5"], "1990-01-01", "2013-01-01"),
    "oli": landsat(["L8", "L9"], "2013-04-01", "2026-01-01"),
}
for name, col in families.items():
    print(name, col.size().getInfo(), "images")

## 2. Illumination through the years (no pixels, one small request)
Landsat 5 crossed the equator later and later after 1995. The sun then stood higher and further
towards the afternoon, so west-facing slopes were lit more directly over the years and
east-facing slopes less.

In [ ]:
import eofigs
import matplotlib.pyplot as plt

eofigs.set_style(journal="custom", font="Arial", bold=True, labels="A")


def sun_table(images):
    """Sun angles and dates of every image (one small request; no pixels)."""
    info = ee.Dictionary(
        {
            "time": images.aggregate_array("system:time_start"),
            "zenith": images.aggregate_array("tc_sun_zenith"),
            "azimuth": images.aggregate_array("tc_sun_azimuth"),
        }
    ).getInfo()
    years = 1970 + np.asarray(info["time"], float) / (365.25 * 86400e3)
    return years, np.asarray(info["zenith"], float), np.asarray(info["azimuth"], float)


def cos_i(zenith, azimuth, slope, aspect):
    z, a, s, e = (np.radians(v) for v in (zenith, azimuth, slope, aspect))
    return np.cos(z) * np.cos(s) + np.sin(z) * np.sin(s) * np.cos(a - e)


def illumination_by_year(series, slope=30.0, months=None, title=""):
    """Yearly median cos i of an east- and a west-facing slope, for each sensor series.

    ``months`` (e.g. (6, 7, 8)) keeps the same season every year, so the change between years
    is not hidden by a changing mix of acquisition dates.
    """
    fig, ax = eofigs.new_figure(width="single", height_mm=60)
    for label, (years, zen, azi) in series.items():
        if months is not None:
            keep = np.isin(np.floor((years % 1) * 12).astype(int) + 1, months)
            years, zen, azi = years[keep], zen[keep], azi[keep]
        year = np.floor(years).astype(int)
        for aspect, style in ((90.0, "-"), (270.0, "--")):
            ci = cos_i(zen, azi, slope, aspect)
            ys = np.unique(year)
            med = [np.median(ci[year == y]) for y in ys]
            side = "east" if aspect == 90.0 else "west"
            ax.plot(ys, med, style, marker="o", ms=2.5, label=f"{label}, {side}-facing")
    ax.set_xlabel("Year")
    season = "" if months is None else f", months {months[0]}-{months[-1]}"
    ax.set_ylabel(f"Median cos i ({slope:.0f}° slope{season})")
    ax.legend(fontsize=6, frameon=False)
    if title:
        ax.set_title(title)
    return fig


series = {"Landsat 5": sun_table(families["l5"]), "Landsat 8-9": sun_table(families["oli"])}
fig = illumination_by_year(series, months=(6, 7, 8))  # the same season every year
eofigs.save_figure(fig, OUT / "fig_illumination_by_year")
plt.show()

## 3. Stack models
One model per sensor family: c as a smooth function of solar zenith, per band and stratum
(WorldCover land cover split by canopy density, x slope class). Progress is printed as
`images done / total`; an interrupted fit continues from its cache.

In [ ]:
models = {}
for name, col in families.items():
    path = OUT / f"{SITE}_{name}_model.json"
    if path.exists():
        models[name] = topocorr.load_model(path)
        print(name, "model loaded from", path)
        continue
    model = topocorr.fit(
        col, strata="worldcover-density", region=aoi, cache=OUT / f"{SITE}_{name}_fit_cache.json"
    )
    model.to_json(path)
    models[name] = model
for name, model in models.items():
    print(name, "sun on both sides of the tile:", model.sun_both_sides)
print(models["l5"].summary())

## 4. Drift maps (`diagnose`) and the method comparison (`compare_export`)
All are batch exports to Google Drive, started once. `diagnose` splits the observed NIRv trend
into the drift artefact (the trend that the changing illumination alone creates) and the total
illumination effect, and classes the drift risk.

In [ ]:
crs, transform = scene_grid(families["oli"])
map_families = ["l5", "oli"] if MAP_OLI else ["l5"]
for name in map_families:
    key = f"{SITE}_diagnose_{INDEX}_{name}"

    def make(name=name, key=key):
        image = topocorr.diagnose(families[name], models[name], index=INDEX).toFloat()
        return ee.batch.Export.image.toDrive(
            image=image,
            description=key,
            folder=DRIVE_FOLDER,
            fileNamePrefix=key,
            region=aoi,
            crs=crs,
            crsTransform=transform,
            maxPixels=1e9,
        )

    start_once(key, make)

compare_manifest = OUT / f"{SITE}_compare_oli.json"
if not compare_manifest.exists():
    topocorr.compare_export(
        families["oli"],
        models["oli"],
        region=aoi,
        n_points=500,
        folder=DRIVE_FOLDER,
        prefix=f"{SITE}_compare_oli",
        manifest=compare_manifest,
    )
    tasks = load_tasks()
    for part, task_id in json.loads(compare_manifest.read_text())["tasks"].items():
        tasks[f"{SITE}_compare_oli_{part}"] = {"id": task_id, "state": "READY"}
    TASKS.write_text(json.dumps(tasks, indent=1), encoding="utf-8")
else:
    print("compare export already started:", compare_manifest)

In [ ]:
wait_for_tasks()  # safe to interrupt and re-run; the tasks keep running in Earth Engine

In [ ]:
files = [f"{SITE}_diagnose_{INDEX}_{name}.tif" for name in map_families]
files += [f"{SITE}_compare_oli_raw.csv", f"{SITE}_compare_oli_points.csv"]
fetch(files)

## 5. Figures: drift trend and risk class
The drift trend is in NIRv units per year; the map shows it per decade.

In [ ]:
import rioxarray  # noqa: F401  (registers .rio)
import xarray as xr

drift_map = xr.open_dataarray(OUT / f"{SITE}_diagnose_{INDEX}_l5.tif", engine="rasterio")
bands = list(drift_map.attrs.get("long_name", []))


def band(name):
    return drift_map.sel(band=bands.index(name) + 1).drop_vars("band")


fig = eofigs.trend_map(
    slope=band("drift_trend") * 10,
    variable=INDEX,
    exclude=None,
    aoi_label="Kalam tile (24 km)",
    width="single",
)
print(fig.eofigs_layout["caption"])
eofigs.save_figure(fig, OUT / "fig_drift_trend_l5")
plt.show()

RISK = {0: "not assessable", 1: "low", 2: "medium", 3: "high", 4: "no trend"}  # risk_class values
risk = band("risk_class").values
values, counts = np.unique(risk[np.isfinite(risk)], return_counts=True)
for v, n in zip(values, counts, strict=True):
    print(f"{RISK[int(v)]:>15}: {100 * n / counts.sum():.1f} % of pixels")

## 6. Which correction? (`compare`, local: no Earth Engine)
Per method: the remaining dependence of NIR on illumination, the sunlit-shaded gap, temporal
noise, values out of range and the **predicted drift left (from the stack model)**, which
assumes the model is right (the data-based check is `diagnose`). Takes about 30 s.

In [ ]:
table = topocorr.compare(compare_manifest)
table